# Manual comparison: released GPT-2 Lens vs no-framing skip-2 Lens

Edit the text, layer, and inspected token below. Both lenses are evaluated through the public `lens.analyze(...)` API with the same native, unframed input and shared model/tokenizer. Component IDs are not aligned between independently fitted lenses; the final cell uses the saved Hungarian direction matching when comparing individual components.

In [ ]:
from pathlib import Path
import csv

import matplotlib.pyplot as plt
import numpy as np
import torch
from gb10_load_llm import load_model_to_cuda
from transformers import AutoModelForCausalLM, AutoTokenizer

from icalens import ICALens

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').exists())
RELEASED_PATH = ROOT / 'local-icalens-models/official/icalens-gpt2-small-pile10k'
SKIP2_PATH = ROOT / 'experiments/initial-token-exclusion/runs/gpt2-no-framing-skip-2'
RESULTS_ROOT = ROOT / 'experiments/initial-token-exclusion/results'

released = ICALens.from_pretrained(RELEASED_PATH)
skip2 = ICALens.from_pretrained(SKIP2_PATH)
assert released.model_id == skip2.model_id
assert released.model_revision == skip2.model_revision
print('Model:', released.model_id, released.model_revision)
print('Available layers:', released.available_layers)

## Load GPT-2 once on the GB10

This uses `gb10-load-llm`; it does not use Accelerate or `device_map`.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    released.model_id, revision=released.model_revision, use_fast=True
)
model = load_model_to_cuda(
    AutoModelForCausalLM,
    released.model_id,
    revision=released.model_revision,
    device='cuda',
    dtype=torch.bfloat16,
    touch='auto',
    low_cpu_mem_usage=True,
)
model.eval()
print('Loaded on', next(model.parameters()).device)

## Choose a prompt and layer

No BOS/EOS token is prepended. Try natural prose, code, non-English text, or a chat-formatted string.

In [ ]:
TEXT = '''The history of science is full of ideas that initially appeared strange,
but later became essential parts of how we understand the world.'''
LAYER = 6
INSPECT_POSITION = 0
TOP_COMPONENTS = 12

assert LAYER in released.available_layers
results = {
    'released': released.analyze(
        TEXT, layer=LAYER, model=model, tokenizer=tokenizer, document_framing='none'
    ),
    'skip-2': skip2.analyze(
        TEXT, layer=LAYER, model=model, tokenizer=tokenizer, document_framing='none'
    ),
}
tokens = list(results['released'].tokens)
assert tuple(tokens) == results['skip-2'].tokens
scores = {name: result.scores.float().cpu().numpy() for name, result in results.items()}
energy = {name: result.energy.float().cpu().numpy() for name, result in results.items()}
print('Tokens:', len(tokens))
print(list(enumerate(tokens)))

## Aggregate score magnitude and component-energy concentration by position

Score RMS reveals broad magnitude shifts. Maximum component-energy share reveals whether the shift is concentrated in one component or spread across many.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
x = np.arange(len(tokens))
for name, values in scores.items():
    rms = np.sqrt(np.mean(values ** 2, axis=1))
    max_share = energy[name].max(axis=1)
    axes[0].plot(x, rms, 'o-', label=name)
    axes[1].plot(x, max_share, 'o-', label=name)
axes[0].set(title=f'Layer {LAYER}: score RMS', xlabel='Token position', ylabel='RMS')
axes[1].set(title=f'Layer {LAYER}: maximum component energy share', xlabel='Token position', ylabel='Share')
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend()
plt.show()

## Inspect the strongest components at one token

The two lists use each lens's own component numbering.

In [ ]:
assert 0 <= INSPECT_POSITION < len(tokens)
print(f'Position {INSPECT_POSITION}: token={tokens[INSPECT_POSITION]!r}')
for name, values in scores.items():
    row = values[INSPECT_POSITION]
    order = np.argsort(np.abs(row))[::-1][:TOP_COMPONENTS]
    print(f'\n{name}')
    print('component   signed score   |score|')
    for component in order:
        print(f'{component:9d} {row[component]:14.5f} {abs(row[component]):9.5f}')

## Compare direction-matched components

This maps released component IDs to the closest one-to-one skip-2 directions. Scores may have opposite signs because ICA direction signs are arbitrary.

In [ ]:
matching_path = RESULTS_ROOT / f'gpt2-layer-{LAYER}/matched-components.csv'
if not matching_path.exists():
    raise FileNotFoundError(f'Run compare_directions.py for layer {LAYER}: {matching_path}')
with matching_path.open() as handle:
    matching = {int(row['reference_component']): row for row in csv.DictReader(handle)}
released_row = scores['released'][INSPECT_POSITION]
skip_row = scores['skip-2'][INSPECT_POSITION]
order = np.argsort(np.abs(released_row))[::-1][:TOP_COMPONENTS]
print('released  skip-2   |direction cosine|   released score   skip-2 score')
for reference_component in order:
    match = matching[int(reference_component)]
    candidate_component = int(match['candidate_component'])
    cosine = float(match['absolute_cosine_similarity'])
    print(
        f'{reference_component:8d} {candidate_component:7d} {cosine:20.4f}'
        f' {released_row[reference_component]:16.5f} {skip_row[candidate_component]:14.5f}'
    )